In [5]:
import dai
import pandas as pd
import numpy as np

def main(data_source=None, start_date='2019-01-01', end_date='2024-12-31'):
    """
    BigAlpha 2026 - Institutional-Grade Multi-Factor Alpha 
    """
    # ============================================
    # 1. 提取分钟行情与 10 档盘口数据
    # ============================================
    bar_fields = []
    for i in range(1, 11):
        bar_fields.extend([
            f'bid_price{i}', f'ask_price{i}',
            f'bid_volume{i}', f'ask_volume{i}',
            f'bid_num_orders{i}', f'ask_num_orders{i}'
        ])
    
    sql_bar = f"""
    SELECT
        date, instrument,
        open, high, low, price, pre_close,
        volume, amount, num_trades,
        total_bid_volume, total_ask_volume,
        bid_avg_price, ask_avg_price,
        {', '.join(bar_fields)}
    FROM bigalpha_2026_stock_bar1m
    WHERE date >= '{start_date}' AND date <= '{end_date}'
    """
    
    # 兼容处理：如果传入了包含 read 的对象则调用，否则统一使用全局 dai.query
    if hasattr(data_source, 'read'):
        df = data_source.read(sql_bar)
    else:
        df = dai.query(sql_bar).df()
    
    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    # ============================================
    # 2. 微观结构特征工程
    # ============================================
    
    # 2a. 10 档指数衰减权重
    raw_weights = np.array([2.0 ** (-i) for i in range(10)])
    weights = raw_weights / raw_weights.sum()
    
    # 2b. OBI (Order Book Imbalance) 加权不平衡度
    obi_vol = np.zeros(len(df))
    for i in range(1, 11):
        w = weights[i - 1]
        bid_v = df[f'bid_volume{i}'].fillna(0)
        ask_v = df[f'ask_volume{i}'].fillna(0)
        obi_vol += w * (bid_v - ask_v) / (bid_v + ask_v + 1.0)
    df['obi'] = obi_vol
    
    # VPIN 知情交易与价格-盘口背离
    df['price_diff'] = df.groupby('instrument')['price'].diff().fillna(0)
    df['obi_price_divergence'] = df['obi'] * np.sign(-df['price_diff'])
    
    # 10档隐蔽撤单率 (Order Cancellation Rate)
    df['bid1_shift'] = df.groupby('instrument')['bid_volume1'].shift(1).fillna(0)
    df['ask1_shift'] = df.groupby('instrument')['ask_volume1'].shift(1).fillna(0)
    df['bid1_cancel'] = (df['bid1_shift'] - df['bid_volume1'] - df['volume']).clip(lower=0)
    df['ask1_cancel'] = (df['ask1_shift'] - df['ask_volume1'] - df['volume']).clip(lower=0)
    df['cancel_ratio_diff'] = (df['ask1_cancel'] - df['bid1_cancel']) / (df['volume'] + 1e-5)

    # 2c. 盘口深度斜率
    bid_slope = np.zeros(len(df))
    ask_slope = np.zeros(len(df))
    for i in range(2, 11):
        bid_slope += (df[f'bid_price{i}'] - df[f'bid_price{i-1}']) / (df[f'bid_price{i-1}'] + 1e-8)
        ask_slope += (df[f'ask_price{i}'] - df[f'ask_price{i-1}']) / (df[f'ask_price{i-1}'] + 1e-8)
    df['slope_diff'] = (bid_slope - ask_slope) / 9.0
    
    # 2d. K线形态与成交量能
    df['intraday_ret'] = (df['price'] - df['open']) / (df['open'] + 1e-8)
    df['price_position'] = (df['price'] - df['low']) / (df['high'] - df['low'] + 1e-8)
    df['trade_intensity'] = df['num_trades'] / (df['volume'] + 1.0)
    
    # ============================================
    # 3. 日频数据聚合 (日内 240 分钟降维)
    # ============================================
    agg_dict = {
        'obi': ['mean', 'std'],
        'obi_price_divergence': 'mean',
        'cancel_ratio_diff': 'mean',
        'slope_diff': ['mean', 'std'],
        'intraday_ret': ['mean', 'std', 'last'],
        'price_position': 'mean',
        'trade_intensity': 'mean',
        'amount': 'sum',
        'volume': 'sum'
    }
    agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}
    
    daily = df.groupby(['date', 'instrument']).agg(agg_dict)
    daily.columns = ['_'.join(c).strip('_') for c in daily.columns]
    daily = daily.reset_index()
    
    # ============================================
    # 4. 生成子信号 + Sigmoid 非线性激活
    # ============================================
    def sigmoid_zscore(series):
        z = (series - series.mean()) / (series.std() + 1e-8)
        return 1.0 / (1.0 + np.exp(-z))

    daily['obi_signal'] = daily['obi_mean'] / (daily['obi_std'] + 1e-8)
    daily['divergence_signal'] = daily['obi_price_divergence_mean']
    daily['cancel_signal'] = daily['cancel_ratio_diff_mean']
    daily['intraday_signal'] = daily['intraday_ret_last'] / (daily['intraday_ret_std'] + 1e-8)
    
    signal_cols = ['obi_signal', 'divergence_signal', 'cancel_signal', 'intraday_signal']
    for col in signal_cols:
        if col in daily.columns:
            daily[col] = daily.groupby('date')[col].transform(sigmoid_zscore)

    # ============================================
    # 5. 财务数据整合
    # ============================================
    try:
        sql_fin = f"""
        SELECT date, instrument,
            operating_revenue, net_profit, total_assets, total_liability
        FROM bigalpha_2026_financial
        WHERE date >= '{start_date}' AND date <= '{end_date}'
        """
        if hasattr(data_source, 'read'):
            fin = data_source.read(sql_fin)
        else:
            fin = dai.query(sql_fin).df()
            
        if not fin.empty and 'total_assets' in fin.columns:
            fin = fin.sort_values(['date', 'instrument']).drop_duplicates(['date', 'instrument'], keep='last')
            fin['roa'] = fin['net_profit'] / (fin['total_assets'] + 1.0)
            fin['debt_ratio'] = fin['total_liability'] / (fin['total_assets'] + 1.0)
            
            daily = daily.merge(fin[['date', 'instrument', 'roa', 'debt_ratio']], on=['date', 'instrument'], how='left')
            daily['fin_signal'] = daily.groupby('date')['roa'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8)).fillna(0) - \
            daily.groupby('date')['debt_ratio'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8)).fillna(0)
        else:
            daily['fin_signal'] = 0
    except Exception:
        daily['fin_signal'] = 0

    # ============================================
    # 6. 截面残差中性化 + 终极因子合成
    # ============================================
    weights_map = {
        'obi_signal': 0.20,
        'divergence_signal': 0.20,
        'cancel_signal': 0.20,
        'intraday_signal': 0.15,
        'fin_signal': 0.25
    }
    
    daily['factor'] = 0.0
    for col, w in weights_map.items():
        if col in daily.columns:
            z = daily.groupby('date')[col].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8)).fillna(0)
            daily['factor'] += w * z
            
    # 剔除成交量偏置
    def neutralize_by_amount(df_day):
        if len(df_day) < 10 or 'amount_sum' not in df_day.columns:
            return df_day['factor']
        x = np.log(df_day['amount_sum'] + 1.0)
        y = df_day['factor']
        cov = np.cov(x, y)[0, 1]
        var = np.var(x)
        beta = cov / (var + 1e-8) if var > 0 else 0
        return y - beta * (x - np.mean(x))

    if 'amount_sum' in daily.columns:
        daily['factor'] = daily.groupby('date', group_keys=False).apply(neutralize_by_amount)

    # 1% / 99% Winsorize
    daily['factor'] = daily.groupby('date')['factor'].transform(
        lambda x: x.clip(x.quantile(0.01), x.quantile(0.99))
    )
    
    # 5日滑动平均平滑
    daily = daily.sort_values(['instrument', 'date'])
    daily['factor'] = daily.groupby('instrument')['factor'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # ============================================
    # 7. 输出 3 列格式
    # ============================================
    result = daily[['date', 'instrument', 'factor']].copy()
    result['date'] = pd.to_datetime(result['date'])
    result = result.dropna(subset=['factor'])
    
    return result
